# Process-Based Phenology Model Exploration

This notebook explores the behaviour of all process-based models implemented in pysephone,
from the temperature response functions themselves through to full season simulations and
sensitivity analyses.

**Structure**
1. Temperature response curves — how each chilling and forcing function responds to temperature
2. Synthetic temperature generation — seasonal series at different winter warmth levels
3. Four-panel season diagnostics — temperature, daily chill/force, cumulative chill, cumulative forcing
4. Season dynamics — temperature bars coloured by chilling/forcing contribution
5. Multi-model comparison — all models on the same season
6. Sensitivity analysis — predicted bloom day vs. mean winter temperature
7. Real data — Japan cherry blossom (GMU yedoensis + sargentii) sample analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from typing import Callable, Dict, List, Optional, Tuple

from pysephone.models.util.func_phenology import (
    func_utah_chill,
    func_chilling_days,
    func_dynamic_chill_daily,
    func_dynamic_chill_hourly,
    func_hourly_from_tmean,
)

In [ ]:
# ---------------------------------------------------------------------------
# Shared style
# ---------------------------------------------------------------------------

PALETTE = {
    'temp_warm':  '#f4a46280',
    'temp_cold':  '#7eb8d480',
    'temp_line':  '#c0392b',
    'chill':      '#2980b9',
    'force':      '#d35400',
    'dechill':    '#c0392b',
    'none':       '#d5d5d5',
    'zero':       '#cccccc',
    'thresh':     '#555555',
}

MODEL_COLORS = {
    'GDD':                '#888888',
    'Utah+GDD':           '#2980b9',
    'ChillingDays+GDD':   '#27ae60',
    'Dynamic+GDD':        '#8e44ad',
}

# Month tick marks for a season starting Oct 1 (day 0 = Oct 1)
_MONTH_STARTS = [0, 31, 61, 92, 122, 153, 181, 212, 243, 273, 304, 334]
_MONTH_LABELS = ['Oct','Nov','Dec','Jan','Feb','Mar',
                  'Apr','May','Jun','Jul','Aug','Sep']

def _month_ticks(n_days: int):
    ticks  = [d for d in _MONTH_STARTS if d < n_days]
    labels = [_MONTH_LABELS[i] for i, d in enumerate(_MONTH_STARTS) if d < n_days]
    return ticks, labels

def _style_ax(ax, n_days, ylabel='', xlabel=False, ylim=None):
    ax.set_xlim(0, n_days - 1)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ticks, labels = _month_ticks(n_days)
    ax.set_xticks(ticks)
    ax.set_xticklabels(labels if xlabel else [], fontsize=8)
    if xlabel:
        ax.set_xlabel('Month (season starting Oct 1)', fontsize=9)
    if ylim is not None:
        ax.set_ylim(ylim)

## 1. Temperature response functions

Each chilling model translates a daily (or hourly) temperature into a chill unit contribution.
Here we plot the response of each function over the temperature range relevant for chilling (−5 to 25 °C).

In [ ]:
def dynamic_steady_state_response(
    temps_range: np.ndarray,
    amplitude: float = 5.0,
    n_days: int = 60,
) -> np.ndarray:
    """Steady-state daily chill portions for each constant temperature.

    Run the Dynamic model at each temperature for *n_days* and take the mean
    of the last 10 days to approximate the quasi-steady-state response.
    """
    result = []
    for t in temps_range:
        ts = np.full(n_days, float(t))
        daily = func_dynamic_chill_daily(ts, amplitude=amplitude)
        result.append(float(daily[-10:].mean()))
    return np.array(result)

In [ ]:
temp_range = np.linspace(-5, 25, 300)

utah_resp  = func_utah_chill(temp_range)
cd_resp    = func_chilling_days(temp_range, t_threshold=7.2)
dyn_resp_a5  = dynamic_steady_state_response(temp_range, amplitude=5.0)
dyn_resp_a10 = dynamic_steady_state_response(temp_range, amplitude=10.0)
gdd_resp_b5  = np.maximum(temp_range - 5.0, 0.0)    # forcing, t_base=5

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Temperature response functions', fontsize=12, fontweight='bold')

# --- Left: chilling responses ---
ax = axes[0]
ax.axhline(0, color=PALETTE['zero'], lw=0.8)
ax.axvline(0, color=PALETTE['zero'], lw=0.8, ls=':')

ax.plot(temp_range, utah_resp,    color=MODEL_COLORS['Utah+GDD'],
        lw=2.2, label='Utah chill (daily, non-monotonic)')
ax.plot(temp_range, cd_resp,      color=MODEL_COLORS['ChillingDays+GDD'],
        lw=2.2, label='Chilling days  (≤ 7.2 °C = 1, else 0)')
ax.plot(temp_range, dyn_resp_a5,  color=MODEL_COLORS['Dynamic+GDD'],
        lw=2.2, label='Dynamic model  (amplitude = 5 °C)')
ax.plot(temp_range, dyn_resp_a10, color=MODEL_COLORS['Dynamic+GDD'],
        lw=2.2, ls='--', alpha=0.6, label='Dynamic model  (amplitude = 10 °C)')

# annotate Utah zones
for x, lbl in [(0.7, '0'), (1.9, '0.5'), (5.5, '1'), (10.5, '0.5'),
               (12.5, '0'), (17, '−0.5'), (21, '−1')]:
    ax.text(x, func_utah_chill(np.array([x], dtype=float))[0] + 0.04, lbl,
            fontsize=6.5, color=MODEL_COLORS['Utah+GDD'], ha='center')

ax.set_xlabel('Daily mean temperature (°C)', fontsize=9)
ax.set_ylabel('Chill units / day', fontsize=9)
ax.set_title('Chilling models', fontsize=10)
ax.legend(fontsize=8, framealpha=0.85)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# --- Right: GDD forcing + inset comparing shapes ---
ax = axes[1]
for t_base, alpha, label in [(3, 0.5, 't_base = 3 °C'),
                              (5, 1.0, 't_base = 5 °C'),
                              (7, 0.5, 't_base = 7 °C')]:
    ax.plot(temp_range, np.maximum(temp_range - t_base, 0),
            color=PALETTE['force'], lw=2.0, alpha=alpha, label=label)

ax.axvline(5, color=PALETTE['force'], lw=0.8, ls=':', alpha=0.5)
ax.set_xlabel('Daily mean temperature (°C)', fontsize=9)
ax.set_ylabel('GDU / day  (forcing units)', fontsize=9)
ax.set_title('Forcing model — GDU  =  max(T − t_base, 0)', fontsize=10)
ax.legend(fontsize=8, framealpha=0.85)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### 1b. Utah model — zoomed piecewise detail

The Utah model uses a piecewise lookup table rather than a smooth curve.
Temperatures above ~18 °C contribute *negative* chill units (de-chilling).

In [ ]:
t_fine = np.linspace(-2, 25, 500)
utah_fine = func_utah_chill(t_fine)

fig, ax = plt.subplots(figsize=(10, 3.5))

# Fill positive (chilling) and negative (de-chilling) separately
ax.fill_between(t_fine, utah_fine, 0,
                where=(utah_fine > 0),
                color=PALETTE['chill'], alpha=0.25, label='Chilling')
ax.fill_between(t_fine, utah_fine, 0,
                where=(utah_fine < 0),
                color=PALETTE['dechill'], alpha=0.25, label='De-chilling')
ax.plot(t_fine, utah_fine, color=MODEL_COLORS['Utah+GDD'], lw=2.0)
ax.axhline(0, color=PALETTE['zero'], lw=0.8)

# Utah bin boundaries
boundaries = [1.4, 2.4, 9.1, 12.4, 15.9, 18.0]
values     = [0.0, 0.5, 1.0, 0.5, 0.0, -0.5, -1.0]
for b in boundaries:
    ax.axvline(b, color='grey', lw=0.6, ls='--', alpha=0.5)
for i, (b, v) in enumerate(zip([0] + boundaries + [25], values)):
    ax.text((([0] + boundaries + [25])[i] + ([0] + boundaries + [25])[i+1]) / 2
            if i < len(values) else 22,
            v + 0.06 * (1 if v >= 0 else -1),
            str(v), fontsize=8, ha='center',
            color=MODEL_COLORS['Utah+GDD'])

ax.set_xlabel('Temperature (°C)', fontsize=9)
ax.set_ylabel('Utah chill units / day', fontsize=9)
ax.set_title('Utah chill model — piecewise response', fontsize=11, fontweight='bold')
ax.legend(fontsize=9, framealpha=0.85)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(-2, 25)
plt.tight_layout()
plt.show()

### 1c. Dynamic model — effect of diurnal amplitude

The Dynamic model needs hourly temperatures. When only daily means are available,
they are reconstructed with a cosine cycle of half-amplitude *A*.
Larger amplitude → the daily minimum is colder → potentially more chill portions.

In [ ]:
t_range = np.linspace(-5, 20, 200)
amplitudes = [2.5, 5.0, 8.0, 12.0]

fig, ax = plt.subplots(figsize=(9, 4))
for amp in amplitudes:
    resp = dynamic_steady_state_response(t_range, amplitude=amp, n_days=60)
    ax.plot(t_range, resp, lw=2.0, label=f'amplitude = {amp} °C')

ax.axhline(0, color=PALETTE['zero'], lw=0.8)
ax.set_xlabel('Daily mean temperature (°C)', fontsize=9)
ax.set_ylabel('Chill portions / day  (steady-state)', fontsize=9)
ax.set_title('Dynamic chill model — steady-state response vs. diurnal amplitude',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9, framealpha=0.85)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 2. Synthetic temperature series

We generate synthetic seasonal temperature series using a cosine seasonal cycle
with Gaussian day-to-day noise, starting on Oct 1 (day 0) and running for 365 days.

In [ ]:
def generate_temperature_series(
    n_days: int = 365,
    mean_winter_temp: float = 4.0,
    mean_summer_temp: float = 20.0,
    noise_std: float = 3.0,
    seed: Optional[int] = None,
) -> np.ndarray:
    """Synthetic seasonal daily mean temperature starting on Oct 1.

    The cosine minimum is placed at day 92 (~Jan 1) so the season
    opens with autumn cooling and warms through spring.
    """
    rng = np.random.default_rng(seed)
    mid_winter_day = 92
    days = np.arange(n_days)
    phase = 2 * np.pi * (days - mid_winter_day) / 365.0
    amplitude = (mean_summer_temp - mean_winter_temp) / 2.0
    baseline  = (mean_summer_temp + mean_winter_temp) / 2.0
    seasonal  = baseline - amplitude * np.cos(phase)
    return (seasonal + rng.normal(0, noise_std, size=n_days)).astype(np.float32)

In [ ]:
SCENARIOS = [
    (0.0,  'Very cold winter  (0 °C)'),
    (4.0,  'Typical winter    (4 °C)'),
    (8.0,  'Mild winter       (8 °C)'),
    (12.0, 'Warm winter      (12 °C)'),
]

temps_dict = {
    label: generate_temperature_series(mean_winter_temp=mw, noise_std=2.0, seed=42)
    for mw, label in SCENARIOS
}

n_days = 365
days   = np.arange(n_days)

fig, axes = plt.subplots(len(SCENARIOS), 1, figsize=(12, 8), sharex=True)
fig.suptitle('Synthetic temperature series — varying winter warmth',
             fontsize=12, fontweight='bold')

for ax, (mw, label) in zip(axes, SCENARIOS):
    ts = temps_dict[label]
    ax.axhline(0, color=PALETTE['zero'], lw=0.8, zorder=1)
    ax.fill_between(days, ts, 0, where=(ts >= 0),
                    color=PALETTE['temp_warm'], zorder=2)
    ax.fill_between(days, ts, 0, where=(ts < 0),
                    color=PALETTE['temp_cold'], zorder=2)
    ax.plot(days, ts, color=PALETTE['temp_line'], lw=1.0, zorder=3)
    ax.set_ylabel('T (°C)', fontsize=8)
    ax.set_title(label, fontsize=9, pad=2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

_style_ax(axes[-1], n_days, xlabel=True)
plt.tight_layout()
plt.show()

## 3. Season response helper

A lightweight function that applies any chill function to a temperature series
and returns all quantities needed for diagnostic plots.

In [ ]:
def compute_season_response(
    temps: np.ndarray,
    chill_fn: Callable,
    t_base: float,
    threshold_c: float,
    threshold_f: float,
) -> dict:
    """Apply a chill-forcing model to a temperature series.

    Args:
        temps:       Daily mean temperatures (°C).
        chill_fn:    Callable ``temps -> chill_units`` (e.g. func_utah_chill).
                     Pass ``None`` for GDD-only (no chilling gate).
        t_base:      Forcing base temperature (°C).
        threshold_c: Chilling requirement (chill units).
        threshold_f: Forcing requirement (GDU).

    Returns:
        Dict with all daily and cumulative arrays plus event indices.
    """
    n = len(temps)

    if chill_fn is not None:
        chill_daily = chill_fn(temps)
    else:
        chill_daily = np.zeros(n)

    force_daily = np.maximum(temps - t_base, 0.0)
    chill_cum   = np.cumsum(chill_daily)

    # Chilling fulfillment
    fulfill = n
    if chill_fn is not None:
        idx = np.where(chill_cum >= threshold_c)[0]
        if len(idx):
            fulfill = int(idx[0])
    else:
        fulfill = 0  # GDD-only: no chilling gate

    force_masked = force_daily.copy()
    force_masked[:fulfill] = 0.0
    force_cum = np.cumsum(force_masked)

    # Flowering
    flower = n
    idx2 = np.where(force_cum >= threshold_f)[0]
    if len(idx2):
        flower = int(idx2[0])

    return dict(
        chill_daily   = chill_daily,
        force_daily   = force_daily,
        force_masked  = force_masked,
        chill_cum     = chill_cum,
        force_cum     = force_cum,
        chill_fulfill = fulfill,
        flower_day    = flower,
        occurred      = flower < n,
        threshold_c   = threshold_c,
        threshold_f   = threshold_f,
    )


# Model definitions for exploration — literature / default parameters
MODELS = {
    'GDD': dict(
        chill_fn    = None,
        t_base      = 5.0,
        threshold_c = 0.0,
        threshold_f = 200.0,
        color       = MODEL_COLORS['GDD'],
        label       = 'GDD  (no chilling gate)',
    ),
    'Utah+GDD': dict(
        chill_fn    = func_utah_chill,
        t_base      = 5.0,
        threshold_c = 50.0,
        threshold_f = 200.0,
        color       = MODEL_COLORS['Utah+GDD'],
        label       = 'Utah+GDD',
    ),
    'ChillingDays+GDD': dict(
        chill_fn    = lambda ts: func_chilling_days(ts, t_threshold=7.2),
        t_base      = 5.0,
        threshold_c = 50.0,
        threshold_f = 200.0,
        color       = MODEL_COLORS['ChillingDays+GDD'],
        label       = 'ChillingDays+GDD  (7.2 °C)',
    ),
    'Dynamic+GDD': dict(
        chill_fn    = lambda ts: func_dynamic_chill_daily(ts, amplitude=5.0),
        t_base      = 5.0,
        threshold_c = 30.0,
        threshold_f = 200.0,
        color       = MODEL_COLORS['Dynamic+GDD'],
        label       = 'Dynamic+GDD  (amp = 5 °C)',
    ),
}

## 4. Four-panel season diagnostics

For a single model and season this shows:
1. Temperature with chilling/forcing phase shading
2. Daily chill unit contributions
3. Daily forcing unit contributions (grey = before chilling gate opens)
4. Normalised cumulative chill and forcing, with threshold crossings

In [ ]:
def plot_season_diagnostics(
    temps: np.ndarray,
    resp: dict,
    model_name: str = '',
    title: str = '',
    show: bool = True,
) -> plt.Figure:
    """Four-panel diagnostic plot for a single season / model.

    Args:
        temps:      Daily mean temperatures.
        resp:       Output of :func:`compute_season_response`.
        model_name: Short label for the model.
        title:      Figure title.
        show:       Call plt.show().
    """
    n     = len(temps)
    days  = np.arange(n)
    cf    = resp['chill_fulfill']
    fd    = resp['flower_day']
    occ   = resp['occurred']
    th_c  = resp['threshold_c']
    th_f  = resp['threshold_f']

    chill_norm = resp['chill_cum'] / th_c if th_c > 0 else np.zeros(n)
    force_norm = resp['force_cum'] / th_f

    fig = plt.figure(figsize=(12, 9))
    fig.suptitle(title or f'Season diagnostics — {model_name}',
                 fontsize=13, fontweight='bold', y=0.98)
    gs = gridspec.GridSpec(4, 1, hspace=0.08,
                           top=0.93, bottom=0.07, left=0.08, right=0.93)
    ax_t = fig.add_subplot(gs[0])
    ax_c = fig.add_subplot(gs[1])
    ax_f = fig.add_subplot(gs[2])
    ax_s = fig.add_subplot(gs[3])

    # ----- Panel 1: temperature -----------------------------------------
    ax_t.axhline(0, color=PALETTE['zero'], lw=0.8, zorder=1)
    ax_t.fill_between(days, temps, 0, where=(temps >= 0),
                      color=PALETTE['temp_warm'], zorder=2)
    ax_t.fill_between(days, temps, 0, where=(temps < 0),
                      color=PALETTE['temp_cold'], zorder=2)
    ax_t.plot(days, temps, color=PALETTE['temp_line'], lw=1.1, zorder=3)
    if cf < n:
        ax_t.axvspan(0, cf, color=PALETTE['chill'], alpha=0.07, zorder=0)
    if occ:
        ax_t.axvspan(cf, fd, color=PALETTE['force'], alpha=0.07, zorder=0)
    _style_ax(ax_t, n, ylabel='Temp (°C)')
    ax_t.legend(handles=[
        mpatches.Patch(color=PALETTE['chill'], alpha=0.4, label='Chilling phase'),
        mpatches.Patch(color=PALETTE['force'], alpha=0.4, label='Forcing phase'),
    ], fontsize=8, loc='upper right', framealpha=0.8)

    # ----- Panel 2: daily chill -----------------------------------------
    pos = np.maximum(resp['chill_daily'], 0)
    neg = np.minimum(resp['chill_daily'], 0)
    ax_c.bar(days, pos, width=1.0, color=PALETTE['chill'], alpha=0.75, zorder=2)
    if np.any(neg < 0):
        ax_c.bar(days, neg, width=1.0, color=PALETTE['dechill'], alpha=0.75, zorder=2)
        ax_c.axhline(0, color=PALETTE['zero'], lw=0.8)
    if cf < n:
        ax_c.axvline(cf, color=PALETTE['chill'], lw=1.4, ls='--', zorder=5)
    _style_ax(ax_c, n, ylabel='Chill / day')
    ax_c.legend(handles=[
        mpatches.Patch(color=PALETTE['chill'], alpha=0.8, label='Chill units'),
        mpatches.Patch(color=PALETTE['dechill'], alpha=0.8, label='De-chill (Utah)'),
    ], fontsize=8, loc='upper right', framealpha=0.8)

    # ----- Panel 3: daily forcing ---------------------------------------
    ax_f.bar(days, resp['force_daily'], width=1.0,
             color=PALETTE['force'], alpha=0.2, zorder=2, label='Potential forcing')
    ax_f.bar(days, resp['force_masked'], width=1.0,
             color=PALETTE['force'], alpha=0.85, zorder=3, label='Active forcing')
    if cf < n:
        ax_f.axvline(cf, color=PALETTE['chill'], lw=1.4, ls='--', zorder=5)
    if occ:
        ax_f.axvline(fd, color=PALETTE['force'], lw=1.4, ls='--', zorder=5)
    _style_ax(ax_f, n, ylabel='GDU / day')
    ax_f.legend(fontsize=8, loc='upper left', framealpha=0.8)

    # ----- Panel 4: normalised cumulatives ------------------------------
    if th_c > 0:
        ax_s.plot(days, chill_norm, color=PALETTE['chill'], lw=2.0, zorder=3,
                  label='Cumul. chill (norm.)')
        ax_s.fill_between(days, chill_norm, color=PALETTE['chill'],
                          alpha=0.12, zorder=2)
    ax_s.plot(days, force_norm, color=PALETTE['force'], lw=2.0, zorder=3,
              label='Cumul. forcing (norm.)')
    ax_s.fill_between(days, force_norm, color=PALETTE['force'],
                      alpha=0.12, zorder=2)
    ax_s.axhline(1.0, color=PALETTE['thresh'], lw=1.0, ls=':',
                 label='Requirement threshold')

    if cf < n:
        ax_s.axvline(cf, color=PALETTE['chill'], lw=1.4, ls='--', zorder=5)
        ax_s.annotate(f'Chilling\nday {cf}',
                      xy=(cf, 1.0), xytext=(cf + 8, 1.3), fontsize=7.5,
                      color=PALETTE['chill'],
                      arrowprops=dict(arrowstyle='->', color=PALETTE['chill'], lw=0.8))
    if occ:
        ax_s.axvline(fd, color=PALETTE['force'], lw=1.4, ls='--', zorder=5)
        ax_s.annotate(f'Bloom\nday {fd}',
                      xy=(fd, 1.0), xytext=(fd + 8, 0.65), fontsize=7.5,
                      color=PALETTE['force'],
                      arrowprops=dict(arrowstyle='->', color=PALETTE['force'], lw=0.8))
    else:
        ax_s.text(0.98, 0.5, 'Bloom did\nnot occur',
                  transform=ax_s.transAxes, ha='right', va='center',
                  fontsize=8, color=PALETTE['force'],
                  bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                            edgecolor=PALETTE['force'], alpha=0.8))

    _style_ax(ax_s, n, ylabel='Cumul. (norm.)',
              xlabel=True, ylim=(-0.05, 1.7))
    ax_s.legend(fontsize=8, loc='upper left', framealpha=0.8)

    # Shared vertical event lines
    for ax in [ax_t, ax_c, ax_f]:
        if cf < n:
            ax.axvline(cf, color=PALETTE['chill'], lw=1.0, ls='--', alpha=0.35)
        if occ:
            ax.axvline(fd, color=PALETTE['force'], lw=1.0, ls='--', alpha=0.35)

    fig.legend(handles=[
        Line2D([0],[0], color=PALETTE['chill'], lw=1.5, ls='--',
               label=f'Chilling fulfilled (day {cf})'),
        Line2D([0],[0], color=PALETTE['force'], lw=1.5, ls='--',
               label=f'Bloom (day {fd})' if occ else 'Bloom did not occur'),
    ], loc='lower center', ncol=2, fontsize=8.5,
       framealpha=0.85, bbox_to_anchor=(0.5, 0.005))

    if show:
        plt.show()
    return fig

In [ ]:
# Show Utah+GDD at three winter temperatures
for mw, label in [(0.0, 'Very cold winter  (0 °C)'),
                  (4.0, 'Typical winter    (4 °C)'),
                  (10.0, 'Warm winter      (10 °C)')]:
    ts   = generate_temperature_series(mean_winter_temp=mw, noise_std=2.0, seed=42)
    m    = MODELS['Utah+GDD']
    resp = compute_season_response(ts, m['chill_fn'], m['t_base'],
                                   m['threshold_c'], m['threshold_f'])
    plot_season_diagnostics(
        ts, resp,
        model_name='Utah+GDD',
        title=f'Utah+GDD diagnostics — {label}',
    )

## 5. Season dynamics — temperature bars coloured by contribution

Each day's temperature bar is coloured by the phase and contribution:
- **Blue** (chilling phase): darker = more effective chilling; **red** = de-chilling (Utah)
- **Orange** (forcing phase): darker = more GDU
- **Grey**: zero contribution

In [ ]:
def _bar_colours(
    temps: np.ndarray,
    chill_daily: np.ndarray,
    force_daily: np.ndarray,
    chill_fulfill: int,
) -> list:
    n = len(temps)
    ca = np.abs(chill_daily[:chill_fulfill])
    cmax = ca.max() if len(ca) > 0 and ca.max() > 0 else 1.0
    fa   = force_daily[chill_fulfill:]
    fmax = fa.max() if len(fa) > 0 and fa.max() > 0 else 1.0

    colours = []
    for i in range(n):
        if i < chill_fulfill:
            v = chill_daily[i]
            if v > 0.01:
                colours.append(cm.Blues(0.3 + 0.65 * v / cmax))
            elif v < -0.01:
                colours.append((0.78, 0.1, 0.1, 0.35 + 0.6 * abs(v) / cmax))
            else:
                colours.append(mcolors.to_rgba(PALETTE['none']))
        else:
            v = force_daily[i]
            if v > 0.01:
                colours.append(cm.Oranges(0.3 + 0.65 * v / fmax))
            else:
                colours.append(mcolors.to_rgba(PALETTE['none']))
    return colours


def plot_season_dynamics(
    temps: np.ndarray,
    model_name: str,
    chill_fn,
    t_base: float,
    threshold_c: float,
    threshold_f: float,
    title: str = '',
    show: bool = True,
) -> plt.Figure:
    """Temperature bars coloured by chilling/forcing contribution + cumulatives."""
    resp  = compute_season_response(temps, chill_fn, t_base, threshold_c, threshold_f)
    n     = len(temps)
    days  = np.arange(n)
    cf    = resp['chill_fulfill']
    fd    = resp['flower_day']
    occ   = resp['occurred']

    bar_cols = _bar_colours(temps, resp['chill_daily'],
                            resp['force_daily'], cf)
    chill_norm = resp['chill_cum'] / threshold_c if threshold_c > 0 else np.zeros(n)
    force_norm = resp['force_cum'] / threshold_f

    fig = plt.figure(figsize=(13, 9))
    fig.suptitle(title or f'Season dynamics — {model_name}',
                 fontsize=12, fontweight='bold', y=0.98)
    gs = gridspec.GridSpec(3, 1, hspace=0.1,
                           top=0.93, bottom=0.08, left=0.08, right=0.94)
    ax_t = fig.add_subplot(gs[0])
    ax_d = fig.add_subplot(gs[1])
    ax_s = fig.add_subplot(gs[2])

    # ----- Coloured temperature bars ------------------------------------
    ax_t.bar(days, temps, width=1.0, color=bar_cols, align='center', zorder=2)
    ax_t.axhline(0, color=PALETTE['zero'], lw=0.8, zorder=1)
    if cf < n:
        ax_t.axvspan(0, cf, color=PALETTE['chill'], alpha=0.04, zorder=0)
    if occ:
        ax_t.axvspan(cf, fd, color=PALETTE['force'], alpha=0.04, zorder=0)
    if cf < n:
        ax_t.axvline(cf, color=PALETTE['chill'], lw=1.4, ls='--', alpha=0.8)
    if occ:
        ax_t.axvline(fd, color=PALETTE['force'], lw=1.4, ls='--', alpha=0.8)
    _style_ax(ax_t, n, ylabel='Temp (°C)')
    leg = [
        mpatches.Patch(facecolor=cm.Blues(0.75),   label='Chilling (effective)'),
        mpatches.Patch(facecolor=cm.Oranges(0.75), label='Forcing (active)'),
        mpatches.Patch(facecolor=PALETTE['none'],  label='No contribution'),
    ]
    if np.any(resp['chill_daily'] < 0):
        leg.insert(1, mpatches.Patch(facecolor=(0.78, 0.1, 0.1, 0.7),
                                     label='De-chilling (Utah)'))
    ax_t.legend(handles=leg, fontsize=8, loc='upper right',
                framealpha=0.85, ncol=len(leg))

    # ----- Daily contributions (twin axis) ------------------------------
    ax_d2 = ax_d.twinx()
    pos = np.maximum(resp['chill_daily'], 0)
    neg = np.minimum(resp['chill_daily'], 0)
    ax_d.bar(days, pos, width=1.0, color=PALETTE['chill'], alpha=0.75, zorder=2)
    if np.any(neg < 0):
        ax_d.bar(days, neg, width=1.0, color=PALETTE['dechill'],
                 alpha=0.75, zorder=2)
        ax_d.axhline(0, color=PALETTE['zero'], lw=0.8)
    ax_d2.bar(days, resp['force_masked'], width=1.0,
              color=PALETTE['force'], alpha=0.75, zorder=2)
    if cf < n:
        ax_d.axvline(cf, color=PALETTE['chill'], lw=1.4, ls='--', alpha=0.7)
    if occ:
        ax_d.axvline(fd, color=PALETTE['force'], lw=1.4, ls='--', alpha=0.7)
    _style_ax(ax_d, n, ylabel='Chill units / day')
    ax_d2.set_ylabel('GDU / day', fontsize=9)
    ax_d2.tick_params(labelsize=8)
    ax_d2.spines['top'].set_visible(False)
    ax_d.legend(handles=[
        mpatches.Patch(color=PALETTE['chill'], alpha=0.8, label='Daily chill'),
        mpatches.Patch(color=PALETTE['force'], alpha=0.8, label='Daily forcing'),
    ], fontsize=8, loc='upper right', framealpha=0.8)

    # ----- Normalised cumulatives ---------------------------------------
    if threshold_c > 0:
        ax_s.plot(days, chill_norm, color=PALETTE['chill'], lw=2.0, zorder=3,
                  label='Cumul. chill (norm.)')
        ax_s.fill_between(days, chill_norm, color=PALETTE['chill'],
                          alpha=0.12, zorder=2)
    ax_s.plot(days, force_norm, color=PALETTE['force'], lw=2.0, zorder=3,
              label='Cumul. forcing (norm.)')
    ax_s.fill_between(days, force_norm, color=PALETTE['force'],
                      alpha=0.12, zorder=2)
    ax_s.axhline(1.0, color=PALETTE['thresh'], lw=1.0, ls=':',
                 label='Requirement')
    if cf < n:
        ax_s.axvline(cf, color=PALETTE['chill'], lw=1.4, ls='--')
        ax_s.annotate(f'Chilling\nday {cf}', xy=(cf, 1.0),
                      xytext=(cf+8, 1.35), fontsize=7.5, color=PALETTE['chill'],
                      arrowprops=dict(arrowstyle='->', color=PALETTE['chill'], lw=0.8))
    if occ:
        ax_s.axvline(fd, color=PALETTE['force'], lw=1.4, ls='--')
        ax_s.annotate(f'Bloom\nday {fd}', xy=(fd, 1.0),
                      xytext=(fd+8, 0.65), fontsize=7.5, color=PALETTE['force'],
                      arrowprops=dict(arrowstyle='->', color=PALETTE['force'], lw=0.8))

    _style_ax(ax_s, n, ylabel='Cumul. (norm.)',
              xlabel=True, ylim=(-0.05, 1.65))
    ax_s.legend(fontsize=8, loc='upper left', framealpha=0.8)

    if show:
        plt.show()
    return fig

In [ ]:
# Show all four models on the same season (typical winter, 4 °C)
ts_typical = generate_temperature_series(mean_winter_temp=4.0, noise_std=2.0, seed=42)

for mname, m in MODELS.items():
    plot_season_dynamics(
        ts_typical, mname,
        chill_fn    = m['chill_fn'],
        t_base      = m['t_base'],
        threshold_c = m['threshold_c'],
        threshold_f = m['threshold_f'],
        title       = f'Season dynamics — {mname}  |  mean winter = 4 °C',
    )

## 6. Multi-model comparison on the same season

All models shown on shared axes so differences in timing and accumulation
are directly visible.  The bottom row shows per-model temperature bar colourings.

In [ ]:
def plot_model_comparison(
    temps: np.ndarray,
    models: dict = None,
    title: str = '',
    show: bool = True,
) -> plt.Figure:
    """Overlay all models; per-model temperature bar rows at the bottom.

    Args:
        temps:  Daily mean temperatures.
        models: Dict of {name: model_spec}, defaults to MODELS.
        title:  Figure title.
        show:   Call plt.show().
    """
    if models is None:
        models = MODELS

    n      = len(temps)
    days   = np.arange(n)
    names  = list(models.keys())
    nm     = len(names)

    # Pre-compute all responses
    resps = {
        name: compute_season_response(
            temps, m['chill_fn'], m['t_base'],
            m['threshold_c'], m['threshold_f'],
        )
        for name, m in models.items()
    }

    fig = plt.figure(figsize=(13, 11))
    fig.suptitle(title or 'Model comparison — same temperature series',
                 fontsize=12, fontweight='bold', y=0.98)

    gs = gridspec.GridSpec(
        4, nm, hspace=0.12, wspace=0.06,
        top=0.93, bottom=0.07, left=0.09, right=0.97,
    )
    ax_cd = fig.add_subplot(gs[0, :])   # daily chill
    ax_cc = fig.add_subplot(gs[1, :])   # cumulative chill
    ax_fc = fig.add_subplot(gs[2, :])   # cumulative forcing
    ax_ts = [fig.add_subplot(gs[3, i]) for i in range(nm)]

    # ----- Row 1: daily chill contributions (overlaid) ------------------
    for name, m in models.items():
        if m['chill_fn'] is None:
            continue
        cd = resps[name]['chill_daily']
        ax_cd.plot(days, cd, color=m['color'], lw=1.6, alpha=0.85,
                   label=m['label'], zorder=3)
        ax_cd.fill_between(days, cd, 0,
                           where=(cd > 0), color=m['color'], alpha=0.08)
        if np.any(cd < 0):
            ax_cd.fill_between(days, cd, 0,
                               where=(cd < 0), color=PALETTE['dechill'], alpha=0.1)
    ax_cd.axhline(0, color=PALETTE['zero'], lw=0.8)
    _style_ax(ax_cd, n, ylabel='Chill units / day')
    ax_cd.legend(fontsize=8.5, loc='upper right', framealpha=0.85)
    ax_cd.set_title('Daily chilling contribution', fontsize=9, pad=3)

    # ----- Row 2: normalised cumulative chill ---------------------------
    for name, m in models.items():
        r  = resps[name]
        th = m['threshold_c']
        if th <= 0 or m['chill_fn'] is None:
            continue
        cc = r['chill_cum'] / th
        ax_cc.plot(days, cc, color=m['color'], lw=2.0, alpha=0.9,
                   label=m['label'], zorder=3)
        cf = r['chill_fulfill']
        if cf < n:
            ax_cc.axvline(cf, color=m['color'], lw=0.9, ls=':', alpha=0.7)
            ax_cc.scatter([cf], [1.0], color=m['color'], s=35, zorder=5)

    ax_cc.axhline(1.0, color=PALETTE['thresh'], lw=1.1, ls='--',
                  label='Requirement', zorder=2)
    _style_ax(ax_cc, n, ylabel='Cumul. chill\n(norm.)')
    ax_cc.legend(fontsize=8, loc='upper left', framealpha=0.85)

    # ----- Row 3: normalised cumulative forcing -------------------------
    for i, (name, m) in enumerate(models.items()):
        r  = resps[name]
        fc = r['force_cum'] / m['threshold_f']
        ax_fc.plot(days, fc, color=m['color'], lw=2.0, alpha=0.9,
                   label=m['label'], zorder=3)
        ax_fc.fill_between(days, fc, color=m['color'], alpha=0.07)
        fd = r['flower_day']
        if r['occurred']:
            ax_fc.axvline(fd, color=m['color'], lw=0.9, ls=':', alpha=0.7)
            ax_fc.scatter([fd], [1.0], color=m['color'], s=35, zorder=5)
            ax_fc.annotate(f'{name}\nday {fd}',
                           xy=(fd, 1.0),
                           xytext=(fd + 5, 1.1 + i * 0.12),
                           fontsize=7, color=m['color'],
                           arrowprops=dict(arrowstyle='->',
                                           color=m['color'], lw=0.7))

    ax_fc.axhline(1.0, color=PALETTE['thresh'], lw=1.1, ls='--',
                  label='Requirement', zorder=2)
    _style_ax(ax_fc, n, ylabel='Cumul. forcing\n(norm.)', ylim=(-0.05, 1.65))
    ax_fc.legend(fontsize=8, loc='upper left', framealpha=0.85)

    # ----- Row 4: per-model coloured temperature bars -------------------
    for i, (name, m) in enumerate(models.items()):
        r    = resps[name]
        cols = _bar_colours(temps, r['chill_daily'],
                            r['force_daily'], r['chill_fulfill'])
        ax   = ax_ts[i]
        ax.bar(days, temps, width=1.0, color=cols, align='center', zorder=2)
        ax.axhline(0, color=PALETTE['zero'], lw=0.8, zorder=1)
        cf = r['chill_fulfill']
        fd = r['flower_day']
        if cf < n:
            ax.axvline(cf, color=m['color'], lw=1.3, ls='--', alpha=0.8)
        if r['occurred']:
            ax.axvline(fd, color=PALETTE['force'], lw=1.3, ls='--', alpha=0.8)
        ax.set_title(name, fontsize=8.5, fontweight='bold',
                     color=m['color'], pad=3)
        _style_ax(ax, n,
                  ylabel='Temp (°C)' if i == 0 else '',
                  xlabel=True)
        if i > 0:
            ax.set_yticklabels([])

    fig.legend(handles=[
        mpatches.Patch(facecolor=cm.Blues(0.75),   label='Chilling'),
        mpatches.Patch(facecolor=cm.Oranges(0.75), label='Forcing'),
        mpatches.Patch(facecolor=PALETTE['none'],  label='No contrib.'),
        mpatches.Patch(facecolor=(0.78,0.1,0.1,0.7), label='De-chill'),
    ], loc='lower center', ncol=4, fontsize=8,
       framealpha=0.85, bbox_to_anchor=(0.5, 0.005))

    if show:
        plt.show()
    return fig

In [ ]:
for mw, label in [(4.0, 'typical winter (4 °C)'),
                  (10.0, 'warm winter (10 °C)')]:
    ts = generate_temperature_series(mean_winter_temp=mw, noise_std=2.0, seed=42)
    plot_model_comparison(
        ts,
        title=f'Model comparison — mean winter = {mw} °C  |  {label}',
    )

## 7. Sensitivity analysis — bloom day vs. winter temperature

We sweep the mean winter temperature from 0 to 16 °C and show how the predicted
bloom day varies for each model.  We also show when chilling is fulfilled.

Each point is the median over `N_SEEDS` synthetic seasons (different noise realisations)
to smooth out noise effects.

In [ ]:
WINTER_TEMPS = np.arange(0.0, 17.0, 0.5)
N_SEEDS      = 10

sensitivity = {name: {'bloom': [], 'chill': []} for name in MODELS}

for mw in WINTER_TEMPS:
    for name, m in MODELS.items():
        blooms = []
        chills = []
        for seed in range(N_SEEDS):
            ts = generate_temperature_series(
                mean_winter_temp=mw, noise_std=2.0, seed=seed)
            r = compute_season_response(
                ts, m['chill_fn'], m['t_base'],
                m['threshold_c'], m['threshold_f'])
            blooms.append(r['flower_day'])
            chills.append(r['chill_fulfill'])
        sensitivity[name]['bloom'].append(np.median(blooms))
        sensitivity[name]['chill'].append(np.median(chills))

print('Sensitivity sweep done.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Model sensitivity to winter temperature  (median over 10 noise seeds)',
             fontsize=12, fontweight='bold')

# Month lines (approximate DOY from Oct 1)
for doy, month in [(92, 'Jan'), (153, 'Mar'), (181, 'Apr'), (212, 'May')]:
    for ax in axes:
        ax.axhline(doy, color='lightgrey', lw=0.7, zorder=0)
        ax.text(16.3, doy, month, fontsize=7, va='center', color='grey')

# --- Left: bloom day ---
ax = axes[0]
for name, m in MODELS.items():
    bloom = np.array(sensitivity[name]['bloom'])
    # Grey out days == 365 (bloom did not occur)
    did_not_occur = bloom >= 364
    ax.plot(WINTER_TEMPS[~did_not_occur], bloom[~did_not_occur],
            color=m['color'], lw=2.2, label=m['label'])
    if np.any(did_not_occur):
        ax.axvline(WINTER_TEMPS[~did_not_occur][-1],
                   color=m['color'], lw=0.8, ls=':', alpha=0.5)

ax.set_xlabel('Mean winter temperature (°C)', fontsize=9)
ax.set_ylabel('Predicted bloom day  (day of season from Oct 1)', fontsize=9)
ax.set_title('Predicted bloom day', fontsize=10)
ax.legend(fontsize=8, framealpha=0.85)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, 16.5)

# --- Right: chilling fulfillment day ---
ax = axes[1]
for name, m in MODELS.items():
    if m['chill_fn'] is None:
        continue
    chill = np.array(sensitivity[name]['chill'])
    not_met = chill >= 364
    ax.plot(WINTER_TEMPS[~not_met], chill[~not_met],
            color=m['color'], lw=2.2, label=m['label'])
    if np.any(not_met):
        ax.axvline(WINTER_TEMPS[~not_met][-1],
                   color=m['color'], lw=0.8, ls=':', alpha=0.5)
        ax.text(WINTER_TEMPS[~not_met][-1] + 0.2, 300,
                f'{name}\nnot met →', fontsize=6.5,
                color=m['color'], va='top')

ax.set_xlabel('Mean winter temperature (°C)', fontsize=9)
ax.set_ylabel('Chilling fulfillment day', fontsize=9)
ax.set_title('Chilling fulfillment day', fontsize=10)
ax.legend(fontsize=8, framealpha=0.85)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, 16.5)

plt.tight_layout()
plt.show()

## 8. Real temperature data — Japan cherry blossom (GMU yedoensis + sargentii)

Load a handful of representative samples from the Japan cherry blossom dataset
and apply the response functions directly (using literature parameters).
This shows how the same model behaviour plays out on real seasonal temperature data.

In [ ]:
from pysephone.dataset.dataset import Dataset
from pysephone.dataset.util.calendar import Calendar
from pysephone.dataset.util.openmeteo import OpenMeteoFeatures
from pysephone.constants import KEY_FEATURES

cal      = Calendar(default_start='10-01', default_length=365)
features = OpenMeteoFeatures(calendar=cal)

ds_ys = Dataset.load('GMU_Cherry_Japan_YS', calendar=cal, feature_providers=[features])
ds_ys.download_features(verbose=True)

ds_y = ds_ys.select_species([('gmu_cherry', 0)])   # Somei-yoshino (yedoensis)
ds_s = ds_ys.select_species([('gmu_cherry', 1)])   # Sargentii

print(f'Yedoensis: {len(ds_y)} samples | Sargentii: {len(ds_s)} samples')

In [ ]:
ds_y = ds_ys.select_species([('GMU_cherry', 0)])   # Somei-yoshino (yedoensis)
ds_s = ds_ys.select_species([('GMU_cherry', 1)])   # Sargentii

print(f'Yedoensis: {len(ds_y)} samples | Sargentii: {len(ds_s)} samples')

In [ ]:
# Show multi-model comparison on a real temperature series
for sample, species_label in [
    (sample_y_median, 'Somei-yoshino (yedoensis)'),
    (sample_s_median, 'Sargentii'),
]:
    ts = sample[KEY_FEATURES]['temperature_2m_mean'].astype(float)

    # Show raw temperature first
    n   = len(ts)
    days = np.arange(n)
    fig, ax = plt.subplots(figsize=(12, 2.8))
    ax.axhline(0, color=PALETTE['zero'], lw=0.8, zorder=1)
    ax.fill_between(days, ts, 0, where=(ts >= 0),
                    color=PALETTE['temp_warm'], zorder=2)
    ax.fill_between(days, ts, 0, where=(ts < 0),
                    color=PALETTE['temp_cold'], zorder=2)
    ax.plot(days, ts, color=PALETTE['temp_line'], lw=1.0, zorder=3)
    _style_ax(ax, n, ylabel='T (°C)', xlabel=True)
    ax.set_title(f'Real temperature — {species_label}  |  year {sample["year"]}',
                 fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Multi-model comparison
    plot_model_comparison(
        ts,
        title=(f'Model comparison — {species_label}  |  year {sample["year"]}'),
    )

In [ ]:
# --- Chilling model overlay on real data: Utah vs Dynamic ---
# Compare the raw chill-unit trajectories for the two non-trivial models
# on early, median, and late bloom years for Somei-yoshino

samples_y = {}
for target in ('early', 'median', 'late'):
    samples_y[target] = pick_bloom_year_sample(ds_y, target)

fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharex=True)
fig.suptitle('Chill accumulation — Somei-yoshino — early / median / late bloom years',
             fontsize=12, fontweight='bold')

for col, (target, sample) in enumerate(samples_y.items()):
    ts   = sample[KEY_FEATURES]['temperature_2m_mean'].astype(float)
    n    = len(ts)
    days = np.arange(n)

    # Row 0: Temperature
    ax = axes[0, col]
    ax.axhline(0, color=PALETTE['zero'], lw=0.8)
    ax.fill_between(days, ts, 0, where=(ts >= 0), color=PALETTE['temp_warm'])
    ax.fill_between(days, ts, 0, where=(ts < 0),  color=PALETTE['temp_cold'])
    ax.plot(days, ts, color=PALETTE['temp_line'], lw=0.9)
    ax.set_title(f'{target.capitalize()} bloom  (year {sample["year"]})',
                 fontsize=9, fontweight='bold')
    _style_ax(ax, n, ylabel='T (°C)' if col == 0 else '')

    # Row 1: Cumulative Utah vs Dynamic chill
    ax = axes[1, col]
    for name, chill_fn, th_c in [
        ('Utah',    func_utah_chill,                               MODELS['Utah+GDD']['threshold_c']),
        ('Dynamic', lambda t: func_dynamic_chill_daily(t, amplitude=5.0), MODELS['Dynamic+GDD']['threshold_c']),
    ]:
        cd   = chill_fn(ts)
        cum  = np.cumsum(cd)
        norm = cum / th_c
        ax.plot(days, norm, lw=1.8,
                color=MODEL_COLORS[f'{name}+GDD'] if name != 'Dynamic' else MODEL_COLORS['Dynamic+GDD'],
                label=name)
    ax.axhline(1.0, color=PALETTE['thresh'], lw=0.9, ls=':', label='Threshold')
    _style_ax(ax, n, ylabel='Chill (norm.)' if col == 0 else '')
    if col == 2:
        ax.legend(fontsize=7.5, loc='upper left', framealpha=0.8)

    # Row 2: Daily chill contributions
    ax = axes[2, col]
    utah_d = func_utah_chill(ts)
    dyn_d  = func_dynamic_chill_daily(ts, amplitude=5.0)
    ax.plot(days, utah_d, color=MODEL_COLORS['Utah+GDD'],     lw=1.2, alpha=0.8, label='Utah')
    ax.plot(days, dyn_d,  color=MODEL_COLORS['Dynamic+GDD'],  lw=1.2, alpha=0.8, label='Dynamic')
    ax.axhline(0, color=PALETTE['zero'], lw=0.7)
    _style_ax(ax, n, ylabel='Chill/day' if col == 0 else '', xlabel=True)

plt.tight_layout()
plt.show()